<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip_Interativo_BARCHART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
### CÓDIGO ADAPTADO PARA BARCHART - ESTRUTURA ORIGINAL COMPLETA ###

# ========== CÉLULA 1: INSTALAR PLOTLY ==========
### Rodar essa célula somente uma vez ###
#!pip install plotly


# ========== CÉLULA 2: BIBLIOTECA ==========
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

In [35]:
# ========== CÉLULA 3: FORMATO DE DISPLAY ==========
pd.options.display.float_format = '{:,.4f}'.format

In [42]:
# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else:
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [44]:
# ========== CÉLULA 4: ARQUIVO CSV E FUNÇÕES ==========
# Parametros de entrada
filename = '/content/nqz25-volatility-greeks-exp-12_19_25-50-strikes-+_--10-29-2025.csv'

In [45]:
# ========== CÉLULA 5: PREPARAÇÃO DO ARQUIVO ==========
# ADAPTAÇÃO: Ler CSV do Barchart
df_raw = pd.read_csv(filename)
df_raw = df_raw[df_raw['Type'].notna()].copy()

# DEFINA O SPOT PRICE MANUALMENTE (Barchart não inclui no CSV)
spotPrice = 26200.00  # AJUSTE AQUI

fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

# Data de hoje
todayDate = datetime.now()

In [46]:
# ========== CÉLULA 6: PROCESSAR DADOS DO BARCHART ==========
# Limpar colunas
df_raw['Strike'] = df_raw['Strike'].str.replace(',', '').astype(float)
df_raw['IV'] = df_raw['IV'].str.replace('%', '').astype(float) / 100

# IMPORTANTE: Adicionar Open Interest (Barchart não tem)
df_raw['OpenInt'] = 100  # AJUSTE AQUI

# Separar calls e puts
df_calls = df_raw[df_raw['Type'] == 'Call'].copy()
df_puts = df_raw[df_raw['Type'] == 'Put'].copy()

# Criar DataFrame no formato original (linha por strike com call e put)
strikes_unique = sorted(df_raw['Strike'].unique())
data_list = []

# Data de expiração (ajuste conforme seu CSV)
expiration_date = datetime(2025, 12, 19, 16, 0)

for strike in strikes_unique:
    call_row = df_calls[df_calls['Strike'] == strike]
    put_row = df_puts[df_puts['Strike'] == strike]

    row_data = {
        'ExpirationDate': expiration_date,
        'StrikePrice': strike,
        'CallIV': call_row['IV'].values[0] if len(call_row) > 0 else 0,
        'PutIV': put_row['IV'].values[0] if len(put_row) > 0 else 0,
        'CallGamma': call_row['Gamma'].values[0] if len(call_row) > 0 else 0,
        'PutGamma': put_row['Gamma'].values[0] if len(put_row) > 0 else 0,
        'CallOpenInt': call_row['OpenInt'].values[0] if len(call_row) > 0 else 0,
        'PutOpenInt': put_row['OpenInt'].values[0] if len(put_row) > 0 else 0,
        'CallDelta': call_row['Delta'].values[0] if len(call_row) > 0 else 0,
        'PutDelta': put_row['Delta'].values[0] if len(put_row) > 0 else 0,
    }
    data_list.append(row_data)

df = pd.DataFrame(data_list)


In [47]:
# ========== CÉLULA 7: GAMMA - GEX ==========
# ---=== CALCULATE SPOT GAMMA ===---
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values


In [48]:
# ========== CÉLULA 8: GRÁFICO 1 - ABSOLUTE GAMMA EXPOSURE ==========
# Chart 1: Absolute Gamma Exposure
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',
        marker_line_color='black',
        marker_line_width=0.15,
        name='Gamma Exposure'
    )
)

fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(color='red', width=2, dash='dash')
)

fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12, color='black')
)

fig.update_layout(width=1750, height=800)
fig.show()


In [49]:
# ========== CÉLULA 9: GRÁFICO 2 - GAMMA BY CALLS AND PUTS ==========
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])

chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")

fig.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=0,
    x1=spotPrice,
    y1=max(dfAgg['CallGEX'].to_numpy() / 10**9),
    line=dict(color="black", width=2),
    name="SPX Spot:" + str("{:,.0f}".format(spotPrice))
))

fig.update_layout(width=1750, height=800)
fig.show()


In [50]:
# ========== CÉLULA 10: CALCULAR GAMMA PROFILE ==========
# For each spot level, calculate gamma exposure by applying gamma exposure at that strike
levels = np.linspace(fromStrike, toStrike, 60)

# Expiração próxima e mensal
nextExpiry = df['ExpirationDate'].min()
df["isThirdFriday"] = df['ExpirationDate'].apply(isThirdFriday)
thirdFridays = df.loc[df["isThirdFriday"] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

# Para cada nível de preço
for level in levels:
    df['callGammaEx'] = df.apply(lambda row: calcGammaEx(
        level, row['StrikePrice'], row['CallIV'],
        (row['ExpirationDate'] - todayDate).days / 365,
        0, 0, 'call', row['CallOpenInt']
    ), axis=1)

    df['putGammaEx'] = df.apply(lambda row: calcGammaEx(
        level, row['StrikePrice'], row['PutIV'],
        (row['ExpirationDate'] - todayDate).days / 365,
        0, 0, 'put', row['PutOpenInt']
    ), axis=1)

    totalGamma.append((df['callGammaEx'].sum() - df['putGammaEx'].sum()) / 10**9)

    # Ex-Next Expiry
    totalGammaExNext.append(
        (df.loc[df['ExpirationDate'] != nextExpiry, 'callGammaEx'].sum() -
         df.loc[df['ExpirationDate'] != nextExpiry, 'putGammaEx'].sum()) / 10**9
    )

    # Ex-Next Monthly
    totalGammaExFri.append(
        (df.loc[df['ExpirationDate'] != nextMonthlyExp, 'callGammaEx'].sum() -
         df.loc[df['ExpirationDate'] != nextMonthlyExp, 'putGammaEx'].sum()) / 10**9
    )


In [51]:
# ========== CÉLULA 11: GRÁFICO 3 - GAMMA EXPOSURE PROFILE ==========
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(
    title=chartTitle,
    xaxis_title='Index Price',
    yaxis_title='Gamma Exposure ($ billions/1% move)',
    title_font=dict(size=20, family="Arial Black")
)

fig.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(totalGamma),
    x1=spotPrice,
    y1=max(totalGamma),
    line=dict(color="red", width=2, dash="dash")
))

fig.update_layout(width=1750, height=800, plot_bgcolor='white')
fig.show()


In [52]:
# ========== CÉLULA 12: DELTA - DEX ==========
# ---=== CALCULATE SPOT DELTA ===---
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice * 0.01
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice * 0.01

df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6
dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

In [53]:
# ========== CÉLULA 13: GRÁFICO 4 - ABSOLUTE DELTA EXPOSURE ==========
# Chart 4: Absolute Delta Exposure
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',
        marker_line_color='black',
        marker_line_width=0.15,
        name='Delta Exposure'
    )
)

fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(color='red', width=2, dash='dash')
)

fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black'}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12, color='black')
)

fig_delta4.update_layout(width=1750, height=800)
fig_delta4.show()


In [54]:
# ========== CÉLULA 14: GRÁFICO 5 - DELTA BY CALLS AND PUTS ==========
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])

chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")

fig_delta5.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6),
    x1=spotPrice,
    y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6),
    line=dict(color="black", width=2)
))

fig_delta5.update_layout(width=1750, height=800)
fig_delta5.show()


In [55]:
# ========== CÉLULA 15: CALCULAR DELTA PROFILE ==========
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

for level in levels_delta:
    # Recalcular delta para cada nível (simplificado - usar delta fixo)
    totalDelta.append(df['TotalDelta'].sum())
    totalDeltaExNext.append(
        df.loc[df['ExpirationDate'] != nextExpiry, 'TotalDelta'].sum()
    )
    totalDeltaExFri.append(
        df.loc[df['ExpirationDate'] != nextMonthlyExp, 'TotalDelta'].sum()
    )



In [56]:
# ========== CÉLULA 16: GRÁFICO 6 - DELTA EXPOSURE PROFILE ==========
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(
    title=chartTitle_delta6,
    xaxis_title='Index Price',
    yaxis_title='Delta Exposure ($ millions/1% move)',
    title_font=dict(size=20, family="Arial Black")
)

fig_delta6.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(totalDelta),
    x1=spotPrice,
    y1=max(totalDelta),
    line=dict(color="red", width=2, dash="dash")
))

fig_delta6.update_layout(width=1750, height=800, plot_bgcolor='white')
fig_delta6.show()


# ========== RESULTADOS FINAIS ==========
print("\n=== RESULTADOS GAMMA ===")
print(f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn")
print(f"Call Gamma: ${dfAgg['CallGEX'].sum() / 10**9:,.2f} Bn")
print(f"Put Gamma: ${dfAgg['PutGEX'].sum() / 10**9:,.2f} Bn")

print("\n=== RESULTADOS DELTA ===")
print(f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million")
print(f"Call Delta: ${dfAgg_delta['CallDEX'].sum() / 10**6:,.2f} Million")
print(f"Put Delta: ${dfAgg_delta['PutDEX'].sum() / 10**6:,.2f} Million")


=== RESULTADOS GAMMA ===
Total Gamma: $-0.04 Bn
Call Gamma: $0.55 Bn
Put Gamma: $-0.60 Bn

=== RESULTADOS DELTA ===
Total Delta: $34.37 Million
Call Delta: $122.30 Million
Put Delta: $-87.93 Million
